## ytest match and remove duplicates

In [1]:
!pip install pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print(pd.__version__)

3.0.2


In [2]:
# ytest 6 columns = TESTID, SOURCE, CRASH_DATE_TEXT, CRASH_TIME_2, LAT, LON
df = pd.read_csv("ytest.csv")
df.head()

,SOURCE,TESTID,CRASH_DATE_TEXT,CRASH_TIME_2,LAT,LON
0,MassDOT,A-keep,01 01 2022,2:10 AM,42.338918,-71.107283
1,Xsource,A+1min,01 01 2022,2:11 AM,42.338918,-71.107283
2,Esource,B-triple-reverse-order,01 01 2022,7:15 AM,42.356923,-71.119906
3,Fsource,B+1min,01 01 2022,7:16 AM,42.356923,-71.119906
4,MassDOT,Bkeep+10min,01 01 2022,7:25 AM,42.356923,-71.119906


In [3]:
#CRASH_TIME_2 & CRASH_DATE_TEXT columns are of type str
#LAT & LON are of type float64
#The following code ensures time and date are of type datetime and lat and lon are numeric
df["CRASH_TIME"] = pd.to_datetime(df["CRASH_TIME_2"], format="%I:%M %p", errors="coerce").dt.time
df["CRASH_DATE"] = pd.to_datetime(df["CRASH_DATE_TEXT"], errors="coerce")
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')
df = df.drop(columns=["CRASH_DATE_TEXT", "CRASH_TIME_2"])
df.head()

,SOURCE,TESTID,LAT,LON,CRASH_TIME,CRASH_DATE
0,MassDOT,A-keep,42.338918,-71.107283,02:10:00,2022-01-01
1,Xsource,A+1min,42.338918,-71.107283,02:11:00,2022-01-01
2,Esource,B-triple-reverse-order,42.356923,-71.119906,07:15:00,2022-01-01
3,Fsource,B+1min,42.356923,-71.119906,07:16:00,2022-01-01
4,MassDOT,Bkeep+10min,42.356923,-71.119906,07:25:00,2022-01-01


In [4]:
df.info(show_counts=True, memory_usage=True, verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   SOURCE      14 non-null     str           
 1   TESTID      14 non-null     str           
 2   LAT         13 non-null     float64       
 3   LON         13 non-null     float64       
 4   CRASH_TIME  14 non-null     object        
 5   CRASH_DATE  14 non-null     datetime64[us]
dtypes: datetime64[us](1), float64(2), object(1), str(2)
memory usage: 804.0+ bytes


In [5]:
df.shape #number of rows and columns

(14, 6)

# Finding duplicates under different conditions

In [6]:
# Sort data by time so earlier crashes come before later ones
df = df.sort_values("datetime").reset_index(drop=True)

KeyError: 'datetime'

In [ ]:
# This function calculates the approximate Euclidean distance in meters 
# between two geographic coordinates (latitude and longitude)

def distance_meters(lat1, lon1, lat2, lon2):
    return np.sqrt(
        ((lat1 - lat2) * 111_000)**2 +
        ((lon1 - lon2) * 111_000 * np.cos(np.radians(lat1)))**2
    )


In [ ]:
# This gives a score between 0 and 1 that tells us:
# "How likely are these two crashes actually the same event?"
# - 1.0 = almost certainly the same crash
# - 0.0 = very unlikely to be the same crash
# It uses:
# - distance (closer = more likely)
# - time difference (closer in time = more likely)

def compute_confidence(dist, dt, eps_meters, eps_seconds):

    # Convert distance into a score (closer = higher score)
    spatial = 1 - (dist / eps_meters)

    # Convert time difference into a score (closer in time = higher score)
    time = 1 - (dt / eps_seconds)

    # Make sure scores stay between 0 and 1
    spatial = max(0, min(1, spatial))
    time = max(0, min(1, time))

    # Combine both scores 
    # space is prioritized since time is not reliable as discussed
    return 0.7 * spatial + 0.3 * time

In [ ]:
# This function compares crashes and finds pairs that might be duplicates
# Instead of comparing every possible pair,
# we only compare crashes that happen within a specific time window.

def find_duplicates(df, eps_meters, eps_minutes):

    EPS_SECONDS = eps_minutes * 60  # convert minutes to seconds

    duplicate_pairs = []  # this stores matching crash pairs

    n = len(df)

    # Go through each crash one by one
    for i in range(n):
        row_i = df.iloc[i]

        # Look at all future crashes after this one
        for j in range(i + 1, n):
            row_j = df.iloc[j]

            # How far apart in time are the two crashes?
            dt = (row_j["datetime"] - row_i["datetime"]).total_seconds()

            # If too far apart in time, stop checking further
            # (because data is sorted, everything after will be even farther)
            if dt > EPS_SECONDS:
                break

            # We only compare crashes from different data sources
            # (same source duplicates are ignored, but can change if this is a problem too)
            if row_i["SOURCE"] == row_j["SOURCE"]:
                continue

            # Skip rows with missing location data
            if pd.isna(row_i["LAT"]) or pd.isna(row_j["LAT"]):
                continue

            # Compute how far apart they are in space in meters
            dist = distance_meters(
                row_i["LAT"], row_i["LON"],
                row_j["LAT"], row_j["LON"]
            )

            # If they are close enough in space, we consider them a possible duplicate
            if dist <= eps_meters:

                # Compute confidence score: How certain are we this is a duplicate?
                conf = compute_confidence(dist, dt, eps_meters, EPS_SECONDS)

                # Store the pair and all useful information
                duplicate_pairs.append((i, j, dist, dt, conf))

    return duplicate_pairs

In [ ]:
# We try many different threshholds to see how sensitive the results are

# time is in minutes
time_thresholds = [2, 10, 30, 60]

# distance is in meters
# to change the units, the distance_meters function needs to change
distance_thresholds = [5, 30, 50, 100]

results = []

# Try every combination of time & distance thresholds
for t in time_thresholds:
    for d in distance_thresholds:

        pairs = find_duplicates(df, d, t)

        # average confidence helps us understand match quality
        avg_conf = np.mean([p[4] for p in pairs]) if len(pairs) > 0 else 0

        results.append({
            "minutes": t,
            "meters": d,
            "num_pairs": len(pairs),
            "avg_confidence": avg_conf
        })

df_results = pd.DataFrame(results)

print("\nTHRESHOLD TEST RESULTS")
print(df_results.sort_values("num_pairs"))

In [ ]:
# These are the thresholds we decide to use after reviewing results
EPS_MINUTES = 60
EPS_METERS = 100

duplicate_pairs = find_duplicates(df, EPS_METERS, EPS_MINUTES)

In [ ]:
# This creates a dataset where each row contains:
# - crash A
# - crash B
# - distance between them
# - time difference
# - confidence score

pairs = []

for i, j, dist, dt, conf in duplicate_pairs:

    row_i = df.iloc[i].add_prefix("A_")
    row_j = df.iloc[j].add_prefix("B_")

    combined = pd.concat([row_i, row_j])

    combined["distance_m"] = dist
    combined["time_diff_sec"] = dt
    combined["confidence"] = conf

    pairs.append(combined)

df_pairs = pd.DataFrame(pairs).reset_index(drop=True)

In [ ]:
# Creating a clean dataset without duplicates
# We decide which rows to remove so each crash appears only once
to_drop = set()

for i, j, _, _, _ in duplicate_pairs:

    # Prefer keeping MassDOT records when duplicates exist
    if df.loc[j, "SOURCE"] == "MassDOT":
        to_drop.add(i)
    elif df.loc[i, "SOURCE"] == "MassDOT":
        to_drop.add(j)
    else:
        # otherwise just drop one of them
        to_drop.add(j)

df_clean = df.drop(index=to_drop).reset_index(drop=True)


# This is the original dataset with duplicates
df_all = df_short.copy()

In [ ]:
# Summary of results
print("\nFINAL SUMMARY")
print(f"Time threshold     : {EPS_MINUTES} minutes")
print(f"Distance threshold : {EPS_METERS} meters")
print(f"Number of duplicate pairs found : {len(df_pairs)}")
print(f"Average confidence score         : {df_pairs['confidence'].mean() if len(df_pairs) else 0:.3f}")

print("\nDATASET SIZES")
print(f"Original dataset : {len(df_all):,}")
print(f"Clean dataset    : {len(df_clean):,}")
print(f"Duplicate pairs  : {len(df_pairs):,}")

In [ ]:
#abbreviated for ytest
renaming = {
    "TESTID": "testid",
    "SOURCE": "source",
    "datetime": "datetime",
    "LAT": "lat",
    "LON": "lng"
}

df_all = df_all.rename(columns= renaming)
df_clean = df_clean.rename(columns= renaming)
df_pairs = df_pairs.rename(columns= renaming)

In [ ]:
df_all.to_csv("ytest-all-merged.csv")
df_pairs.to_csv("ytest-duplicates-removed.csv")
df_clean.to_csv("ytest-clean.csv")